# Backpatching for Boolean Expressions

Objective: To implement backpatching for Boolean expressions using the operators &&, || and !, and generate intermediate three-address code with labels.

Define Backpatching Functions

In [ ]:
def makelist(i):
    return [i]

def merge(p1, p2):
    return p1 + p2

def backpatch(code, lst, target):
    for i in lst:
        code[i] = code[i].replace("?", str(target))

Define the Boolean Expression Generator

In [ ]:
def generate_code(expr):
    code = []
    temp = 1

    def emit(instruction):
        code.append(instruction)
        return len(code) - 1

    def new_label():
        nonlocal temp
        label = f"L{temp}"
        temp += 1
        return label

    def parse(expr):
        expr = expr.strip()

        if expr.startswith("!"):
            result = parse(expr[1:])
            result["truelist"], result["falselist"] = (
                result["falselist"], result["truelist"]
            )
            return result

        depth = 0
        for i in range(len(expr) - 1):
            if expr[i] == "(":
                depth += 1
            elif expr[i] == ")":
                depth -= 1
            elif depth == 0 and expr[i:i+2] == "||":
                left = parse(expr[:i])
                right_label = new_label()
                backpatch(code, left["falselist"], right_label)
                emit(right_label + ":")
                right = parse(expr[i+2:])
                return {
                    "truelist": merge(left["truelist"], right["truelist"]),
                    "falselist": right["falselist"]
                }

        depth = 0
        for i in range(len(expr) - 1):
            if expr[i] == "(":
                depth += 1
            elif expr[i] == ")":
                depth -= 1
            elif depth == 0 and expr[i:i+2] == "&&":
                left = parse(expr[:i])
                right_label = new_label()
                backpatch(code, left["truelist"], right_label)
                emit(right_label + ":")
                right = parse(expr[i+2:])
                return {
                    "truelist": right["truelist"],
                    "falselist": merge(left["falselist"], right["falselist"])
                }

        if expr.startswith("(") and expr.endswith(")"):
            return parse(expr[1:-1])

        i = emit(f"if {expr} goto ?")
        j = emit("goto ?")
        return {"truelist": makelist(i), "falselist": makelist(j)}

    result = parse(expr)
    true_label = new_label()
    backpatch(code, result["truelist"], true_label)
    emit(true_label + ": TRUE")

    false_label = new_label()
    backpatch(code, result["falselist"], false_label)
    emit(false_label + ": FALSE")

    return code

Enter a Boolean Expression

In [ ]:
expression = input("Enter a Boolean expression: ")
print("Expression:", expression)

Enter a Boolean expression: a && b || !c
Expression: a && b || !c


Generate Intermediate Code

In [ ]:
result = generate_code(expression)

print("Generated Three-Address Code:\n")
for instruction in result:
    print(instruction)

Generated Three-Address Code:

if a goto L1
goto L2
L1:
if b goto L3
goto L2
L2:
if c goto L4
goto L3
L3: TRUE
L4: FALSE


Test Example

In [ ]:
expression = "a && b || !c"

print("Expression:", expression)
print("\nGenerated Intermediate Code:")

for instruction in generate_code(expression):
    print(instruction)

Expression: a && b || !c

Generated Intermediate Code:
if a goto L1
goto L2
L1:
if b goto L3
goto L2
L2:
if c goto L4
goto L3
L3: TRUE
L4: FALSE


Test Another Expression

In [ ]:
expression = "a || b && !c"

print("Expression:", expression)
print("\nGenerated Intermediate Code:")

for instruction in generate_code(expression):
    print(instruction)

Expression: a || b && !c

Generated Intermediate Code:
if a goto L3
goto L1
L1:
if b goto L2
goto L4
L2:
if c goto L4
goto L3
L3: TRUE
L4: FALSE


Algorithm

1. Read the Boolean expression.
2. Generate conditional jumps for atomic Boolean expressions.
3. Maintain `truelist` and `falselist` for incomplete jump targets.
4. For `&&`, backpatch the left expression's truelist to the right expression.
5. For `||`, backpatch the left expression's falselist to the right expression.
6. For `!`, swap the truelist and falselist.
7. Merge pending jump lists where required.
8. Backpatch remaining jumps to the final TRUE and FALSE labels.
9. Display the generated intermediate code.